In [2]:
# Setup the Jupyter version of Dash
from jupyter_dash import JupyterDash

# Configure the necessary Python module imports for dashboard components
import base64
import os

import dash_leaflet as dl
from dash import dcc, html, dash_table
from dash.dependencies import Input, Output
import pandas as pd
import plotly.express as px

JupyterDash.infer_jupyter_proxy_config()

# Import the CRUD module created in Project One
from CRUD_Python_Module import AnimalShelter


###########################
# Data Manipulation / Model
###########################
# Required hard-coded database credentials for the AAC user
username = "aacuser"
password = "stallions"

# Connect to MongoDB through the CRUD module
db = AnimalShelter(username, password)

# Rescue filter queries based on the Dashboard Specifications Document
FILTER_QUERIES = {
    "reset": {},
    "water": {
        "animal_type": "Dog",
        "breed": {
            "$regex": "Labrador Retriever|Chesapeake Bay Retriever|Newfoundland",
            "$options": "i",
        },
        "sex_upon_outcome": "Intact Female",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
    },
    "mountain": {
        "animal_type": "Dog",
        "breed": {
            "$regex": "German Shepherd|Alaskan Malamute|Old English Sheepdog|Siberian Husky|Rottweiler",
            "$options": "i",
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 26, "$lte": 156},
    },
    "disaster": {
        "animal_type": "Dog",
        "breed": {
            "$regex": "Doberman Pinscher|German Shepherd|Golden Retriever|Bloodhound|Rottweiler",
            "$options": "i",
        },
        "sex_upon_outcome": "Intact Male",
        "age_upon_outcome_in_weeks": {"$gte": 20, "$lte": 300},
    },
}

DEFAULT_COLUMNS = [
    "rec_num",
    "age_upon_outcome",
    "animal_id",
    "animal_type",
    "breed",
    "color",
    "date_of_birth",
    "datetime",
    "monthyear",
    "name",
    "outcome_subtype",
    "outcome_type",
    "sex_upon_outcome",
    "location_lat",
    "location_long",
    "age_upon_outcome_in_weeks",
]


def make_dataframe(query):
    """Run a MongoDB query through the CRUD module and return a Dash-safe DataFrame."""
    records = db.read(query)
    frame = pd.DataFrame.from_records(records)

    # MongoDB ObjectId values are not JSON serializable for Dash tables
    if "_id" in frame.columns:
        frame.drop(columns=["_id"], inplace=True)

    # Keep the dashboard layout stable when a filter returns no records
    if frame.empty:
        frame = pd.DataFrame(columns=DEFAULT_COLUMNS)

    return frame


# Initial unfiltered table view
# This is to show an unfiltered Austin Animal Center Outcomes view first
df = make_dataframe(FILTER_QUERIES["reset"])


#########################
# Dashboard Layout / View
#########################
app = JupyterDash(__name__)

# Load the Grazioso Salvare logo
logo_filename = "Grazioso Salvare Logo.png"
if not os.path.exists(logo_filename):
    logo_filename = os.path.join("/mnt/data", "Grazioso Salvare Logo.png")

encoded_logo = base64.b64encode(open(logo_filename, "rb").read()).decode()

app.layout = html.Div([
    html.Div(id="hidden-div", style={"display": "none"}),

html.Center([
    html.Img(
        src="data:image/png;base64,{}".format(encoded_logo),
        style={"height": "180px", "width": "auto"},
    ),
    html.H1("SNHU CS-340 Grazioso Salvare Dashboard"),
    html.H4("Created by Ben Ezekiel"),
]),
html.Hr(),

    html.Div([
        html.H3("Interactive Filter Options"),
        dcc.RadioItems(
            id="filter-type",
            options=[
                {"label": "Water Rescue", "value": "water"},
                {"label": "Mountain or Wilderness Rescue", "value": "mountain"},
                {"label": "Disaster Rescue or Individual Tracking", "value": "disaster"},
                {"label": "Reset", "value": "reset"},
            ],
            value="reset",
            labelStyle={"display": "inline-block", "marginRight": "20px"},
            inputStyle={"marginRight": "6px"},
        ),
    ]),
    html.Hr(),

    dash_table.DataTable(
        id="datatable-id",
        columns=[{"name": i, "id": i, "deletable": False, "selectable": True} for i in df.columns],
        data=df.to_dict("records"),
        editable=False,
        filter_action="native",
        sort_action="native",
        sort_mode="multi",
        column_selectable="single",
        row_selectable="single",
        row_deletable=False,
        selected_columns=[],
        selected_rows=[0],
        page_action="native",
        page_current=0,
        page_size=10,
        style_table={"overflowX": "auto"},
        style_cell={
            "minWidth": "120px",
            "width": "150px",
            "maxWidth": "220px",
            "overflow": "hidden",
            "textOverflow": "ellipsis",
            "textAlign": "left",
        },
        style_header={"fontWeight": "bold"},
    ),

    html.Br(),
    html.Hr(),

    html.Div(
        className="row",
        style={"display": "flex"},
        children=[
            html.Div(id="graph-id", className="col s12 m6", style={"width": "50%"}),
            html.Div(id="map-id", className="col s12 m6", style={"width": "50%"}),
        ],
    ),
])


#############################################
# Interaction Between Components / Controller
#############################################
@app.callback(
    [Output("datatable-id", "data"), Output("datatable-id", "columns"), Output("datatable-id", "selected_rows")],
    [Input("filter-type", "value")],
)
def update_dashboard(filter_type):
    """Update the table from MongoDB when the rescue filter changes."""
    query = FILTER_QUERIES.get(filter_type, FILTER_QUERIES["reset"])
    filtered_df = make_dataframe(query)

    columns = [
        {"name": column, "id": column, "deletable": False, "selectable": True}
        for column in filtered_df.columns
    ]
    data = filtered_df.to_dict("records")

    # Select the first row when data is available so the map has a default marker
    selected_rows = [0] if len(data) > 0 else []
    return data, columns, selected_rows


@app.callback(
    Output("graph-id", "children"),
    [Input("datatable-id", "derived_virtual_data")],
)
def update_graphs(viewData):
    """Display a pie chart of the most common breeds represented in the current table."""
    if viewData is None or len(viewData) == 0:
        return [html.H4("No records to graph for the selected filter.")]

    dff = pd.DataFrame.from_dict(viewData)
    if "breed" not in dff.columns or dff.empty:
        return [html.H4("Breed data is unavailable for the selected filter.")]

    # Limit the chart to ten most common breeds for readability
    breed_counts = dff["breed"].value_counts().nlargest(10).reset_index()
    breed_counts.columns = ["breed", "count"]

    return [
        dcc.Graph(
            figure=px.pie(
                breed_counts,
                names="breed",
                values="count",
                title="Top Breeds in Current Rescue Filter",
            )
        )
    ]


@app.callback(
    Output("datatable-id", "style_data_conditional"),
    [Input("datatable-id", "selected_columns")],
)
def update_styles(selected_columns):
    """Highlight the selected data-table column."""
    return [{
        "if": {"column_id": column},
        "background_color": "#D2F3FF",
    } for column in selected_columns]


@app.callback(
    Output("map-id", "children"),
    [
        Input("datatable-id", "derived_virtual_data"),
        Input("datatable-id", "derived_virtual_selected_rows"),
    ],
)
def update_map(viewData, index):
    """Update the geolocation chart using the selected row in the current table."""
    if viewData is None or len(viewData) == 0:
        return [html.H4("No location data to map for the selected filter.")]

    dff = pd.DataFrame.from_dict(viewData)
    if "location_lat" not in dff.columns or "location_long" not in dff.columns:
        return [html.H4("Location columns are unavailable for the selected filter.")]

    row = 0 if index is None or len(index) == 0 else index[0]
    if row >= len(dff):
        row = 0

    lat = dff.iloc[row].get("location_lat", 30.75)
    lon = dff.iloc[row].get("location_long", -97.48)

    if pd.isna(lat) or pd.isna(lon):
        lat, lon = 30.75, -97.48

    animal_name = dff.iloc[row].get("name", "Unknown")
    breed = dff.iloc[row].get("breed", "Unknown breed")

    return [
        dl.Map(
            style={"width": "100%", "height": "500px"},
            center=[30.75, -97.48],
            zoom=10,
            children=[
                dl.TileLayer(id="base-layer-id"),
                dl.Marker(
                    position=[lat, lon],
                    children=[
                        dl.Tooltip(str(breed)),
                        dl.Popup([
                            html.H3("Animal Name"),
                            html.P(str(animal_name)),
                            html.P(str(breed)),
                        ]),
                    ],
                ),
            ],
        )
    ]


# If port 8050 is unavailable, change the port number below
app.run_server(mode="jupyterlab", port=8050)